# Glacial Lake Segmentation — ResNet34-FCN Baseline (Google Colab Training)

[![Python 3.9+](https://img.shields.io/badge/Python-3.9%2B-blue.svg)](https://www.python.org/)
[![PyTorch 2.0+](https://img.shields.io/badge/PyTorch-2.0%2B-ee4c2c.svg)](https://pytorch.org/)
[![Dataset: GLID](https://img.shields.io/badge/Dataset-GLID-green.svg)](https://doi.org/10.5281/zenodo.14838695)

This notebook is the **dedicated, reproducible training and evaluation pipeline** for the **ResNet-34 Fully Convolutional Network (FCN)** baseline model.

---

### Architectural Specifications
* **Encoder**: ResNet-34 convolutional backbone initialized with ImageNet pretrained weights (`ResNet34_Weights.DEFAULT`).
* **Decoder**: 5-stage progressive transposed-convolution upsampling head ($512 \to 256 \to 128 \to 64 \to 32 \to 16 \to 1$) with Batch Normalization and ReLU activations.
* **Exact Parameters**: **24,079,153 total parameters** (21,284,672 encoder + 2,794,481 decoder; 100% trainable, 0 non-trainable).
* **Comparison Model**: DeepLabV3+ with ResNet-50 + ASPP (**40,347,041 total parameters**).

---

### Key Operational Features
1. **Google Drive Sync & Isolation**: All checkpoints, logs, plots, and evaluation reports are stored in a dedicated folder:
   `Glacial_Lake_Segmentation/experiments/resnet34_fcn_colab/` (completely isolated from DeepLabV3+ runs).
2. **Robust Multi-Path Dataset Discovery**: Automatically scans Google Drive for training (`GLID`) and validation (`val`) image and label directories, supporting nested folder structures.
3. **Local NVMe SSD Caching**: Performs a one-time copy from Drive to Colab local SSD (`/content/dataset/`) for maximum GPU data-feeding throughput.
4. **CUDA Mixed Precision (AMP)**: `torch.cuda.amp` accelerates training on Tesla T4/V100/A100 GPUs while maintaining float32 numerical stability.
5. **Disconnection Resilience & Resumption**: Checkpoints are saved atomically every epoch (`latest_model.pth` and `best_model.pth`), enabling seamless resume without loss of optimizer, scaler, scheduler, or history state.

> **Hardware Setup**: Ensure a GPU runtime is selected via **Runtime → Change runtime type → T4 GPU** (or A100/V100).


## 1. Hardware & Environment Verification
Verify CUDA availability, GPU device specifications, and runtime memory.


In [ ]:
# 1.1 GPU and CUDA Verification
import sys
import torch

print("=" * 65)
print("  Environment & Hardware Verification")
print("=" * 65)
print(f"Python Version    : {sys.version.split()[0]}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU is not detected!\n"
        "Please navigate to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU Model         : {gpu_name}")
print(f"GPU VRAM          : {gpu_mem_gb:.2f} GB")
print(f"CUDA Device Count : {torch.cuda.device_count()}")
print("=" * 65)

# Enable cuDNN benchmark for optimized conv kernels with fixed 512x512 inputs
torch.backends.cudnn.benchmark = True


## 2. Google Drive Mounting & Repository Setup
Mount Google Drive and clone or pull the latest project source code.


In [ ]:
# 2.1 Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print("✓ Google Drive mounted successfully.")


In [ ]:
# 2.2 Clone or update project repository
from pathlib import Path

%cd /content
REPO_DIR = Path("/content/glacial-lake-segmentation")
REPO_URL = "https://github.com/ShubhamSnSharma/glacial-lake-segmentation.git"

if not REPO_DIR.exists():
    print(f"Cloning repository from {REPO_URL}...")
    !git clone -q {REPO_URL}
else:
    print("Repository exists locally. Pulling latest commits from main...")
    %cd {REPO_DIR}
    !git pull -q

%cd {REPO_DIR}
!git log -1 --oneline
print("✓ Working directory set to:", os.getcwd())


In [ ]:
# 2.3 Install project dependencies (preserves Colab's CUDA PyTorch)
!pip install -q pillow pyyaml matplotlib tqdm scikit-learn pandas

import sys
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("✓ Dependencies verified and repository root registered on sys.path.")


## 3. Directory Configuration & Multi-Path Dataset Discovery
Configure experiment directories and search Google Drive for the GLID dataset.


In [ ]:
# 3.1 Path Configuration & Isolated Output Directories
from pathlib import Path

# Base Google Drive root for the project
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Glacial_Lake_Segmentation")

# Dedicated experiment folder for ResNet34-FCN
EXP_DIR = DRIVE_PROJECT_ROOT / "experiments" / "resnet34_fcn_colab"
EXP_CHECKPOINTS = EXP_DIR / "checkpoints"
EXP_RESULTS = EXP_DIR / "results"
EXP_LOGS = EXP_RESULTS / "logs"
EXP_PLOTS = EXP_RESULTS / "plots"
EXP_PREDICTIONS = EXP_RESULTS / "predictions"
EXP_EVALUATION = EXP_RESULTS / "evaluation"
EXP_COMPARISON = EXP_DIR / "comparison"

# Create all persistent experiment directories in Drive
for folder in [EXP_CHECKPOINTS, EXP_LOGS, EXP_PLOTS, EXP_PREDICTIONS, EXP_EVALUATION, EXP_COMPARISON]:
    folder.mkdir(parents=True, exist_ok=True)

# Local fast NVMe SSD storage path
LOCAL_DATA_ROOT = Path("/content/dataset")
LOCAL_TRAIN_DIR = LOCAL_DATA_ROOT / "GLID"
LOCAL_VAL_DIR = LOCAL_DATA_ROOT / "val"

print("✓ Experiment output paths initialized:")
print(f"  Checkpoints : {EXP_CHECKPOINTS}")
print(f"  Logs        : {EXP_LOGS}")
print(f"  Evaluation  : {EXP_EVALUATION}")


In [ ]:
# 3.2 Robust Multi-Path Google Drive Dataset Discovery
import os
import re
from typing import List, Tuple, Optional

def scan_for_dataset_splits(search_roots: List[Path]) -> Tuple[List[Path], List[Path]]:
    """
    Recursively scans search roots for valid GLID training and validation directories.
    A valid directory must contain image and label/mask subdirectories with matching files.
    """
    IMG_CANDIDATES = {"images", "image", "img"}
    LBL_CANDIDATES = {"labels", "label", "masks", "mask", "annotation"}
    
    def check_split_dir(p: Path) -> bool:
        if not p.is_dir():
            return False
        subdirs = {d.name.lower() for d in p.iterdir() if d.is_dir()}
        return bool(subdirs & IMG_CANDIDATES) and bool(subdirs & LBL_CANDIDATES)
    
    train_found, val_found = [], []
    
    for root in search_roots:
        if not root.exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            # Exclude unwanted directories from recursive search
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d not in ["__pycache__", "experiments", "results", "checkpoints"]]
            if check_split_dir(p):
                name_lower = p.name.lower()
                if any(k in name_lower for k in ["glid", "train", "training"]):
                    if p not in train_found:
                        train_found.append(p)
                elif any(k in name_lower for k in ["val", "validation", "test"]):
                    if p not in val_found:
                        val_found.append(p)
                        
    return train_found, val_found

# Define candidate Drive search roots
CANDIDATE_SEARCH_ROOTS = [
    DRIVE_PROJECT_ROOT,
    Path("/content/drive/MyDrive/GLID_dataset"),
    Path("/content/drive/MyDrive/GLID"),
    Path("/content/drive/MyDrive")
]

print("Searching Google Drive for GLID dataset directories...")
discovered_train, discovered_val = scan_for_dataset_splits(CANDIDATE_SEARCH_ROOTS)

print("\n" + "=" * 65)
print("  Discovered Dataset Candidates")
print("=" * 65)
print(f"Training Candidates   : {[str(p) for p in discovered_train]}")
print(f"Validation Candidates : {[str(p) for p in discovered_val]}")
print("=" * 65)

if not discovered_train or not discovered_val:
    raise FileNotFoundError(
        "Could not automatically locate GLID training and validation directories in Google Drive.\n\n"
        "Expected directory structure in Drive:\n"
        "  MyDrive/Glacial_Lake_Segmentation/GLID/images/  and  labels/\n"
        "  MyDrive/Glacial_Lake_Segmentation/val/images/   and  labels/\n\n"
        "If your dataset is located elsewhere, please set `DRIVE_TRAIN_SRC` and `DRIVE_VAL_SRC` manually."
    )

# Select primary candidate (default to first match)
DRIVE_TRAIN_SRC = discovered_train[0]
DRIVE_VAL_SRC = discovered_val[0]

print(f"\n✓ Selected Training Source   : {DRIVE_TRAIN_SRC}")
print(f"✓ Selected Validation Source : {DRIVE_VAL_SRC}")


In [ ]:
# 3.3 Fast One-Time Copy to Local NVMe SSD & Sample Validation
import shutil
import time
from src.dataset import GLIDDataset

# Copy data to Colab NVMe SSD to avoid Google Drive slow I/O bottlenecks during training
LOCAL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

if LOCAL_TRAIN_DIR.exists() and LOCAL_VAL_DIR.exists():
    print("✓ Local SSD dataset cache already present. Skipping copy.")
else:
    print("Copying dataset from Google Drive to local Colab NVMe SSD...")
    t_start = time.time()
    
    if not LOCAL_TRAIN_DIR.exists():
        print(f"  Copying training data: {DRIVE_TRAIN_SRC} -> {LOCAL_TRAIN_DIR} ...")
        shutil.copytree(DRIVE_TRAIN_SRC, LOCAL_TRAIN_DIR)
        
    if not LOCAL_VAL_DIR.exists():
        print(f"  Copying validation data: {DRIVE_VAL_SRC} -> {LOCAL_VAL_DIR} ...")
        shutil.copytree(DRIVE_VAL_SRC, LOCAL_VAL_DIR)
        
    print(f"✓ Copy completed in {(time.time() - t_start) / 60:.1f} minutes.")

# Validate datasets and print discovery counts
train_ds = GLIDDataset(str(LOCAL_TRAIN_DIR), is_train=True, split="train")
val_ds = GLIDDataset(str(LOCAL_VAL_DIR), is_train=False, split="val")

n_train = len(train_ds)
n_val = len(val_ds)

print("\n" + "=" * 65)
print("  GLID Dataset Discovery Verification")
print("=" * 65)
print(f"Training Samples   : {n_train:,} (Expected: 16,000)")
print(f"Validation Samples : {n_val:,} (Expected: 2,367)")
print("=" * 65)

assert n_train > 0, "Training dataset is empty!"
assert n_val > 0, "Validation dataset is empty!"
print("✓ Local dataset successfully loaded and validated.")


## 4. PyTorch DataLoaders & Architecture Initialization


In [ ]:
# 4.1 Initialize PyTorch DataLoaders
from torch.utils.data import DataLoader

BATCH_SIZE = 8          # Set to 4 if encountering CUDA Out Of Memory (OOM)
NUM_WORKERS = 2         # Multi-worker background prefetching
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=2 if NUM_WORKERS > 0 else None,
    drop_last=False,
)

print(f"✓ DataLoaders ready:")
print(f"  Batch size    : {BATCH_SIZE}")
print(f"  Train batches : {len(train_loader):,}")
print(f"  Val batches   : {len(val_loader):,}")


In [ ]:
# 4.2 Verify tensor shape and contract on a real batch
sample_images, sample_masks = next(iter(train_loader))

print("Batch Tensor Check:")
print(f"  Images shape : {sample_images.shape} | dtype: {sample_images.dtype}")
print(f"  Masks shape  : {sample_masks.shape} | dtype: {sample_masks.dtype}")
print(f"  Mask values  : {torch.unique(sample_masks).tolist()}")

assert sample_images.shape == (BATCH_SIZE, 3, 512, 512), "Image shape mismatch!"
assert sample_masks.shape == (BATCH_SIZE, 1, 512, 512), "Mask shape mismatch!"
assert set(torch.unique(sample_masks).tolist()).issubset({0.0, 1.0}), "Masks must be binary {0.0, 1.0}!"
print("✓ Batch shape contract verified.")


In [ ]:
# 4.3 Initialize ResNet34-FCN Baseline Model & Compute Exact Parameter Breakdown
from src.model import build_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fcn_cfg = {
    "model": {
        "name": "resnet34_fcn",
        "pretrained": True,
        "pretrained_source": "imagenet",
        "num_classes": 1,
        "freeze_encoder": False
    }
}

# Instantiate the real training model
model = build_model(fcn_cfg).to(device)
model = model.to(memory_format=torch.channels_last)

# Programmatically compute exact parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)

summary = model.get_parameter_summary()

print("=" * 65)
print("  ResNet34-FCN Verified Parameter Breakdown")
print("=" * 65)
print(f"Total Parameters         : {total_params:,} (100.0%)")
print(f"Trainable Parameters     : {trainable_params:,} (100.0%)")
print(f"Non-Trainable Parameters : {non_trainable_params:,} (0.0%)")
print(f"Encoder (ResNet-34)      : {summary['encoder_parameters']:,} ({summary['encoder_parameters']/total_params*100:.1f}%)")
print(f"Decoder (5-Stage FCN)    : {summary['decoder_parameters']:,} ({summary['decoder_parameters']/total_params*100:.1f}%)")
print("=" * 65)

# Forward pass verification
model.eval()
with torch.no_grad():
    x = sample_images.to(device, memory_format=torch.channels_last)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        out = model(x)

assert out.shape == (BATCH_SIZE, 1, 512, 512), f"Unexpected output shape: {out.shape}"
print(f"✓ Forward pass verified: output shape = {out.shape}")


## 5. GPU Speed Benchmark (Isolated — Zero Model Mutation)
Run a 2-batch benchmark using a **separate temporary model** to measure GPU throughput without altering the training model's weights.


In [ ]:
# 5.1 Isolated Speed Benchmark (Guarantees Training Model Remains 100% Untouched)
import time
import copy
from src.losses import CombinedBCEDiceLoss

# Snapshot training model weights hash to verify zero mutation
initial_weights_snapshot = {k: v.clone().cpu() for k, v in model.state_dict().items()}

print("Running isolated speed benchmark on a temporary model...")

# 1. Create dedicated temporary model and optimizer for benchmark only
bench_model = build_model(fcn_cfg).to(device)
bench_model = bench_model.to(memory_format=torch.channels_last)
bench_model.train()

bench_loss = CombinedBCEDiceLoss()
bench_opt = torch.optim.AdamW(bench_model.parameters(), lr=1e-4)
bench_scaler = torch.amp.GradScaler("cuda")

times = []

for i, (imgs, msks) in enumerate(train_loader):
    if i >= 2:
        break
    t0 = time.time()
    
    imgs = imgs.to(device, non_blocking=True, memory_format=torch.channels_last)
    msks = msks.to(device, non_blocking=True)
    
    bench_opt.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        preds = bench_model(imgs)
        loss = bench_loss(preds, msks)
        
    bench_scaler.scale(loss).backward()
    bench_scaler.unscale_(bench_opt)
    torch.nn.utils.clip_grad_norm_(bench_model.parameters(), max_norm=1.0)
    bench_scaler.step(bench_opt)
    bench_scaler.update()
    
    torch.cuda.synchronize()
    times.append(time.time() - t0)
    print(f"  Benchmark batch {i+1}/2 | Loss: {loss.item():.4f} | Batch Time: {times[-1]:.3f}s")

# Clean up benchmark artifacts and free GPU memory
del bench_model, bench_opt, bench_scaler
torch.cuda.empty_cache()

# 2. Programmatically verify that training model weights are 100% UNTOUCHED
for k, v in model.state_dict().items():
    if not torch.equal(v.cpu(), initial_weights_snapshot[k]):
        raise RuntimeError(f"FATAL: Benchmark mutated training model weights in layer {k}!")

del initial_weights_snapshot

avg_time = sum(times) / len(times)
est_epoch_min = (avg_time * len(train_loader)) / 60

print("\n" + "=" * 65)
print("  GPU Benchmark Results (Zero Model Mutation Verified)")
print("=" * 65)
print(f"Average Batch Time    : {avg_time:.3f} seconds")
print(f"Estimated Epoch Time  : {est_epoch_min:.1f} minutes (~{est_epoch_min * 30 / 60:.1f} hours for 30 epochs)")
print("✓ Verified: Training model weights are 100% intact and unmutated.")
print("=" * 65)


## 6. Checkpoint State Inspection & Resume Configuration
Inspect Google Drive for existing checkpoints to support automatic resumption if a session was previously disconnected.


In [ ]:
# 6.1 Checkpoint Inspection & Resume Configuration
from pathlib import Path

LATEST_CHECKPOINT_PATH = EXP_CHECKPOINTS / "latest_model.pth"
BEST_CHECKPOINT_PATH = EXP_CHECKPOINTS / "best_model.pth"

# Training Hyperparameters (Preserved: 30 epochs, 3 warmup, lr=1e-4, wd=1e-4)
TOTAL_EPOCHS = 30
WARMUP_EPOCHS = 3
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

# Set to True to automatically resume from latest_model.pth if found
RESUME_IF_AVAILABLE = True

print("=" * 65)
print("  Checkpoint Status")
print("=" * 65)
print(f"Latest Checkpoint Path : {LATEST_CHECKPOINT_PATH}")
print(f"Checkpoint Present     : {LATEST_CHECKPOINT_PATH.exists()}")

if LATEST_CHECKPOINT_PATH.exists():
    chk_meta = torch.load(LATEST_CHECKPOINT_PATH, map_location="cpu", weights_only=False)
    print(f"  Last Completed Epoch : {chk_meta.get('epoch', 'N/A')}")
    print(f"  Best Val F1 Recorded : {chk_meta.get('best_metric_val', chk_meta.get('best_f1', 0.0)):.4f}")
    print(f"  History Length       : {len(chk_meta.get('history', []))} epochs")
    print("\n[Status] Ready to resume. Set RESUME_IF_AVAILABLE = True to continue.")
else:
    print("\n[Status] No existing checkpoint found. Training will start fresh from Epoch 1.")


## 7. Resilient 30-Epoch ResNet34-FCN Training Loop
Executes training with live `tqdm` progress, validation metric accumulation, atomic Drive checkpointing, and CSV logging.


In [ ]:
# 7.1 Resilient Training Loop with Live Progress and Atomic Google Drive Sync
import csv
import time
from tqdm.auto import tqdm
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from src.losses import CombinedBCEDiceLoss
from src.metrics import MetricTracker

# 1. Optimizer, Loss Criterion, Scaler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = CombinedBCEDiceLoss(bce_weight=0.5, dice_weight=0.5).to(device)
scaler = torch.amp.GradScaler("cuda")

# 2. Learning Rate Scheduler (Linear Warmup + Cosine Annealing)
if WARMUP_EPOCHS > 0 and TOTAL_EPOCHS > WARMUP_EPOCHS:
    warmup_sched = LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
    cosine_sched = CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])
else:
    scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)

# 3. State variables
start_epoch = 1
best_val_f1 = -1.0
best_epoch = 0
history = []
csv_log_path = EXP_LOGS / "training_history.csv"

# Initialize CSV log file with complete headers
if not csv_log_path.exists():
    with open(csv_log_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch", "train_loss", "train_iou", "train_f1", "train_precision", "train_recall",
            "val_loss", "val_iou", "val_f1", "val_precision", "val_recall", "val_accuracy",
            "learning_rate", "epoch_time"
        ])

# 4. Resume state from checkpoint if requested
if RESUME_IF_AVAILABLE and LATEST_CHECKPOINT_PATH.exists():
    print(f"Loading checkpoint: {LATEST_CHECKPOINT_PATH} ...")
    checkpoint = torch.load(LATEST_CHECKPOINT_PATH, map_location=device, weights_only=False)
    
    model.load_state_dict(checkpoint["model_state_dict"])
    if "optimizer_state_dict" in checkpoint and checkpoint["optimizer_state_dict"] is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"] is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    if "scheduler_state_dict" in checkpoint and checkpoint["scheduler_state_dict"] is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        
    start_epoch = int(checkpoint.get("epoch", 0)) + 1
    best_val_f1 = float(checkpoint.get("best_metric_val", checkpoint.get("best_f1", -1.0)))
    best_epoch = int(checkpoint.get("best_epoch", 0))
    history = list(checkpoint.get("history", []))
    print(f"✓ Resumed successfully: Starting at Epoch {start_epoch}, Best F1 = {best_val_f1:.4f} (Epoch {best_epoch})\n")

# 5. Main Training Execution
print("=" * 75)
print(f"  Training ResNet34-FCN Baseline: Epochs {start_epoch} -> {TOTAL_EPOCHS}")
print("=" * 75)

for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    t_epoch_start = time.time()
    
    # ------------------ TRAINING PASS ------------------
    model.train()
    train_tracker = MetricTracker(threshold=0.5)
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{TOTAL_EPOCHS:02d} [Train]", leave=False)
    
    for images, masks in train_pbar:
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        masks = masks.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, masks)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        batch_m = train_tracker.update(logits.detach(), masks, loss=loss.item())
        train_pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            f1=f"{batch_m['f1']:.3f}",
            iou=f"{batch_m['iou']:.3f}"
        )
        
    train_metrics = train_tracker.compute()
    
    # ------------------ VALIDATION PASS ------------------
    model.eval()
    val_tracker = MetricTracker(threshold=0.5)
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{TOTAL_EPOCHS:02d} [Val]  ", leave=False)
    
    with torch.no_grad():
        for images, masks in val_pbar:
            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            masks = masks.to(device, non_blocking=True)
            
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(images)
                val_loss = criterion(logits, masks)
                
            batch_vm = val_tracker.update(logits, masks, loss=val_loss.item())
            val_pbar.set_postfix(
                val_loss=f"{val_loss.item():.4f}",
                val_f1=f"{batch_vm['f1']:.3f}",
                val_iou=f"{batch_vm['iou']:.3f}"
            )
            
    val_metrics = val_tracker.compute()
    
    # Step scheduler
    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()
    
    epoch_duration = time.time() - t_epoch_start
    is_best = val_metrics["f1"] > best_val_f1
    if is_best:
        best_val_f1 = val_metrics["f1"]
        best_epoch = epoch
        
    # Record history
    record = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_iou": train_metrics["iou"],
        "train_f1": train_metrics["f1"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "val_loss": val_metrics["loss"],
        "val_iou": val_metrics["iou"],
        "val_f1": val_metrics["f1"],
        "val_precision": val_metrics["precision"],
        "val_recall": val_metrics["recall"],
        "val_accuracy": val_metrics["accuracy"],
        "learning_rate": current_lr,
        "epoch_time": round(epoch_duration, 2),
    }
    history.append(record)
    
    # ------------------ ATOMIC CHECKPOINT SAVING TO DRIVE ------------------
    chk_payload = {
        "epoch": epoch,
        "model_name": "resnet34_fcn",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_metric_val": best_val_f1,
        "best_f1": best_val_f1,
        "best_epoch": best_epoch,
        "history": history,
        "batch_size": BATCH_SIZE,
        "amp": True
    }
    
    # Save latest atomically via temporary file
    tmp_latest = EXP_CHECKPOINTS / "latest_model.pth.tmp"
    torch.save(chk_payload, tmp_latest)
    tmp_latest.replace(LATEST_CHECKPOINT_PATH)
    
    # Save best atomically when new best F1 is reached
    if is_best:
        tmp_best = EXP_CHECKPOINTS / "best_model.pth.tmp"
        torch.save(chk_payload, tmp_best)
        tmp_best.replace(BEST_CHECKPOINT_PATH)
        
    # Append to CSV log
    with open(csv_log_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            record["epoch"],
            f"{record['train_loss']:.4f}",
            f"{record['train_iou']:.4f}",
            f"{record['train_f1']:.4f}",
            f"{record['train_precision']:.4f}",
            f"{record['train_recall']:.4f}",
            f"{record['val_loss']:.4f}",
            f"{record['val_iou']:.4f}",
            f"{record['val_f1']:.4f}",
            f"{record['val_precision']:.4f}",
            f"{record['val_recall']:.4f}",
            f"{record['val_accuracy']:.4f}",
            f"{record['learning_rate']:.2e}",
            record["epoch_time"]
        ])
        
    # Epoch summary banner
    remaining_epochs = TOTAL_EPOCHS - epoch
    est_remaining_min = (remaining_epochs * epoch_duration) / 60
    print(
        f"Epoch [{epoch:02d}/{TOTAL_EPOCHS:02d}] ({epoch_duration:.1f}s, ETA: {est_remaining_min:.1f}m) | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val IoU: {val_metrics['iou']:.4f} | "
        f"Prec: {val_metrics['precision']:.4f} | "
        f"Rec: {val_metrics['recall']:.4f} "
        f"{'[★ BEST]' if is_best else ''}"
    )

print("\n" + "=" * 75)
print(f"  Training Run Finished! Best Validation F1 = {best_val_f1:.4f} at Epoch {best_epoch}")
print("=" * 75)


## 8. Training History Visualization


In [ ]:
# 8.1 Plot Loss, F1/Dice Score, and IoU Curves
import pandas as pd
import matplotlib.pyplot as plt

if csv_log_path.exists():
    df_history = pd.read_csv(csv_log_path)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Loss
    axes[0].plot(df_history["epoch"], df_history["train_loss"], label="Train Loss", marker="o", color="#1f77b4")
    axes[0].plot(df_history["epoch"], df_history["val_loss"], label="Val Loss", marker="s", color="#ff7f0e")
    axes[0].set_title("Combined BCE + Dice Loss", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].grid(True, linestyle="--", alpha=0.6)
    axes[0].legend()
    
    # 2. F1 / Dice Score
    axes[1].plot(df_history["epoch"], df_history["train_f1"], label="Train F1", marker="o", color="#1f77b4")
    axes[1].plot(df_history["epoch"], df_history["val_f1"], label="Val F1", marker="s", color="#2ca02c")
    axes[1].set_title("Dice / F1-Score", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("F1 Score")
    axes[1].grid(True, linestyle="--", alpha=0.6)
    axes[1].legend()
    
    # 3. IoU (Jaccard)
    axes[2].plot(df_history["epoch"], df_history["train_iou"], label="Train IoU", marker="o", color="#1f77b4")
    axes[2].plot(df_history["epoch"], df_history["val_iou"], label="Val IoU", marker="s", color="#d62728")
    axes[2].set_title("Intersection over Union (IoU)", fontsize=12, fontweight="bold")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("IoU")
    axes[2].grid(True, linestyle="--", alpha=0.6)
    axes[2].legend()
    
    plt.tight_layout()
    curve_path = EXP_PLOTS / "resnet34_fcn_training_curves.png"
    plt.savefig(curve_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Training curves saved to: {curve_path}")
else:
    print("No training history CSV found to plot.")


## 9. Full Validation Benchmark Evaluation
Execute `scripts/evaluate.py` across all **2,367 validation samples** ($620,494,848$ total pixels) using the best saved ResNet34-FCN checkpoint.


In [ ]:
# 9.1 Full Validation Evaluation via scripts/evaluate.py
import json
import subprocess

print("Running full validation evaluation on best ResNet34-FCN checkpoint...")

eval_cmd = [
    sys.executable, "scripts/evaluate.py",
    "--checkpoint", str(BEST_CHECKPOINT_PATH),
    "--model", "resnet34_fcn",
    "--val-dir", str(LOCAL_VAL_DIR),
    "--output-dir", str(EXP_EVALUATION),
    "--batch-size", "8",
    "--device", "cuda" if torch.cuda.is_available() else "cpu"
]

!{" ".join(eval_cmd)}

# Display evaluation results
fcn_eval_json = EXP_EVALUATION / "eval_metrics.json"

if fcn_eval_json.exists():
    with open(fcn_eval_json) as f:
        eval_data = json.load(f)
    
    m = eval_data["metrics"]
    cm = eval_data["confusion_matrix"]
    print("\n" + "=" * 65)
    print("  ResNet34-FCN Full Validation Benchmark (N=2,367)")
    print("=" * 65)
    print(f"Dice / F1-Score : {m['f1'] * 100:.2f}% ({m['f1']:.6f})")
    print(f"Mean IoU        : {m['iou'] * 100:.2f}% ({m['iou']:.6f})")
    print(f"Precision       : {m['precision'] * 100:.2f}% ({m['precision']:.6f})")
    print(f"Recall          : {m['recall'] * 100:.2f}% ({m['recall']:.6f})")
    print(f"Pixel Accuracy  : {m['accuracy'] * 100:.2f}% ({m['accuracy']:.6f})")
    print("-" * 65)
    print(f"True Positives  : {cm['TP']:,}")
    print(f"False Positives : {cm['FP']:,}")
    print(f"False Negatives : {cm['FN']:,}")
    print(f"True Negatives  : {cm['TN']:,}")
    print("=" * 65)


## 10. DeepLabV3+ vs. ResNet34-FCN Comparative Benchmark

Generate a side-by-side benchmark table comparing the **DeepLabV3+ advanced model** (ResNet-50 + ASPP) against the **ResNet34-FCN baseline** on the identical validation split ($N=2,367$).

> **Academic Rigor & Fairness Notice**:
> - Both architectures were trained and evaluated on the exact same official GLID validation split ($2,367$ samples, $620,494,848$ total pixels).
> - Both architectures use the exact same input standardization (ImageNet $\mu, \sigma$), binary decision threshold ($0.5$), and global confusion matrix metric formulation.
> - All reported numbers represent **validation metrics**, not independent unannotated field test scenes.


In [ ]:
# 10.1 Side-by-Side Architectural Benchmark Comparison
from pathlib import Path
import json

# Locate DeepLabV3+ Checkpoint and Evaluation JSON
dlv3_chk_candidates = [
    DRIVE_PROJECT_ROOT / "training" / "deeplabv3plus_fast" / "checkpoints" / "best_amp_model.pth",
    DRIVE_PROJECT_ROOT / "checkpoints" / "best_amp_model.pth",
    REPO_DIR / "checkpoints" / "best_amp_model.pth"
]
dlv3_eval_candidates = [
    DRIVE_PROJECT_ROOT / "training" / "deeplabv3plus_fast" / "results" / "evaluation" / "eval_metrics.json",
    DRIVE_PROJECT_ROOT / "results" / "evaluation" / "eval_metrics.json",
    REPO_DIR / "results" / "evaluation" / "eval_metrics.json"
]

dlv3_chk = next((p for p in dlv3_chk_candidates if p.is_file()), None)
dlv3_eval = next((p for p in dlv3_eval_candidates if p.is_file()), None)

print(f"DeepLabV3+ Checkpoint  : {dlv3_chk}")
print(f"DeepLabV3+ Eval JSON   : {dlv3_eval}")
print(f"ResNet34-FCN Checkpoint: {BEST_CHECKPOINT_PATH}")
print(f"ResNet34-FCN Eval JSON : {fcn_eval_json}")

if BEST_CHECKPOINT_PATH.exists() and fcn_eval_json.exists():
    compare_cmd = [
        sys.executable, "scripts/compare_models.py",
        "--fcn-checkpoint", str(BEST_CHECKPOINT_PATH),
        "--fcn-eval", str(fcn_eval_json),
        "--output-dir", str(EXP_COMPARISON)
    ]
    if dlv3_chk:
        compare_cmd.extend(["--dlv3-checkpoint", str(dlv3_chk)])
    if dlv3_eval:
        compare_cmd.extend(["--dlv3-eval", str(dlv3_eval)])
        
    !{" ".join(compare_cmd)}
    
    table_md = EXP_COMPARISON / "comparison_table.md"
    if table_md.exists():
        print("\n" + "=" * 65)
        print("  ARCHITECTURAL BENCHMARK COMPARISON TABLE")
        print("=" * 65)
        with open(table_md) as f:
            print(f.read())
else:
    print("ResNet34-FCN evaluation artifacts not found yet. Run training and evaluation first.")


## 11. Qualitative Visual Predictions
Generate 4-panel visual comparison figures (Satellite RGB, Ground Truth, ResNet34-FCN Prediction, Probability Heatmap) for representative validation scenes.


In [ ]:
# 11.1 Generate 4-Panel Qualitative Prediction Visualizations
import matplotlib.pyplot as plt
import numpy as np

model.eval()
vis_loader = DataLoader(val_ds, batch_size=6, shuffle=False)
imgs, msks = next(iter(vis_loader))

with torch.no_grad():
    x = imgs.to(device, memory_format=torch.channels_last)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        logits = model(x)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

imgs_np = imgs.permute(0, 2, 3, 1).cpu().numpy()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
imgs_np = np.clip(imgs_np * std + mean, 0.0, 1.0)

gt_np = msks.squeeze(1).cpu().numpy()
pred_np = preds.squeeze(1).cpu().numpy()
prob_np = probs.squeeze(1).cpu().numpy()

fig, axes = plt.subplots(4, 6, figsize=(18, 12))

for i in range(6):
    # 1. RGB
    axes[0, i].imshow(imgs_np[i])
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_ylabel("Satellite RGB", fontsize=10, fontweight="bold")
        
    # 2. Ground Truth
    axes[1, i].imshow(gt_np[i], cmap="Blues_r", vmin=0, vmax=1)
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_ylabel("Ground Truth", fontsize=10, fontweight="bold")
        
    # 3. ResNet34-FCN Prediction
    axes[2, i].imshow(pred_np[i], cmap="Blues_r", vmin=0, vmax=1)
    axes[2, i].axis("off")
    if i == 0:
        axes[2, i].set_ylabel("ResNet34-FCN Pred", fontsize=10, fontweight="bold")
        
    # 4. Probability Heatmap
    im = axes[3, i].imshow(prob_np[i], cmap="viridis", vmin=0, vmax=1)
    axes[3, i].axis("off")
    if i == 0:
        axes[3, i].set_ylabel("Lake Probability", fontsize=10, fontweight="bold")

plt.suptitle("ResNet34-FCN Baseline Validation Predictions", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
vis_save_path = EXP_PREDICTIONS / "sample_validation_predictions.png"
plt.savefig(vis_save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Qualitative visual predictions saved to: {vis_save_path}")
